In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# timezone for UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs (they are in current working directory)
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_error = pd.read_csv('error_logs.csv')

# convert timestamp to timezone-aware datetime (UTC -> convert to Asia/Shanghai)
for df in (df_metric, df_trace, df_log, df_error):
    if 'timestamp' in df.columns:
        df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Helper to produce aggregated summary for a given dataframe and name-column
def summary_top20(df, name_col):
    grp = df.groupby(['cmdb_id', name_col])['value'].agg(
        count='count',
        min='min',
        max='max',
        mean='mean',
        p50=lambda x: x.quantile(0.5),
        p90=lambda x: x.quantile(0.9),
        p95=lambda x: x.quantile(0.95)
    ).reset_index()
    # round numeric columns for compactness
    num_cols = ['min','max','mean','p50','p90','p95']
    grp[num_cols] = grp[num_cols].round(6)
    grp = grp.sort_values('count', ascending=False).reset_index(drop=True)
    return grp.head(20)

# 1) metric.csv summary top 20 (cmdb_id, kpi_name)
metric_summary = summary_top20(df_metric, 'kpi_name')

# 2) trace.csv summary top 20 (cmdb_id, trace_name)
trace_summary = summary_top20(df_trace, 'trace_name')

# 3) log.csv summary top 20 (cmdb_id, log_name)
log_summary = summary_top20(df_log, 'log_name')

# 4) error_logs.csv: counts per cmdb_id and up to 10 sample rows (timestamp, message) per cmdb_id
error_counts = df_error.groupby('cmdb_id').size().reset_index(name='count').sort_values('count', ascending=False).reset_index(drop=True)

# take up to 10 sample rows per cmdb_id, sort cmdb_ids by descending count for readability
ordered_cmdb = error_counts['cmdb_id'].tolist()
df_error['cmdb_id'] = pd.Categorical(df_error['cmdb_id'], categories=ordered_cmdb, ordered=True)
# keep only requested columns for samples and ensure compactness
error_samples = df_error.sort_values(['cmdb_id','timestamp']).groupby('cmdb_id').head(10)[['cmdb_id','timestamp','dt','message']]
# sort samples by the ordered cmdb_id
error_samples = error_samples.sort_values('cmdb_id').reset_index(drop=True)

# Final compact outputs: metric_summary, trace_summary, log_summary, error_counts, error_samples
metric_summary, trace_summary, log_summary, error_counts, error_samples

```
Out[1]:
```
summary = (
    "Telemetry summary (compact):\n\n"
    "1) metric.csv (top combos by point count):\n"
    "- Top combos have 31 points each. apache02 and apache01 dominate the top entries with OSLinux CPU/process KPIs.\n"
    "- Example metrics: apache02 OSLinux-CPU_CPU_CPULoad (count=31, mean≈0.0203, p50=0.0, p90≈0.07, p95≈0.08); "
    "apache01 OSLinux-CPU_CPU_CPULoad (count=31, mean≈1.072, p50≈1.06, p90≈1.15, p95≈1.195).\n"
    "- Tomcat04 appears with constant MaxThreads (500) across its series (count=31).\n\n"
    "2) trace.csv (top combos by point count):\n"
    "- Top trace combos have 30 points each. dockerA2 and MG01/IG01 related traces appear frequently.\n"
    "- Example traces: dockerA2 trace.self.row_count (count=30, mean≈2211.6, p50=2330, p90≈2781, p95≈3002); "
    "IG01 trace.self.duration_mean (count=30, mean≈0.548, p50≈0.553, p90≈0.709, p95≈0.739).\n\n"
    "3) log.csv (top combos by point count):\n"
    "- Many top log combos have 31 points (apache02, Tomcat01, apache01, Tomcat03, Tomcat04). Some MySQL entries have 28 or fewer.\n"
    "- Example logs: Tomcat01 log.row_count (count=31, mean≈440.3, p50=460, p90≈494, p95≈521.5); "
    "apache02 log.row_count (count=31, mean≈880.1, p50=931, p90≈1019, p95≈1037).\n"
    "- Most log.error_count series shown are zero across their sampled points.\n\n"
    "4) error_logs.csv:\n"
    "- No error_log records were found (error_logs is empty), so there are no cmdb_id samples to display.\n\n"
    "If you want, I can expand any of these top-20 tables into full rows or provide additional percentiles or time-range filtered summaries."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  count         min         max        mean         p50         p90         p95
0   apache02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     31    0.000000    0.000000    0.000000    0.000000    0.000000    0.000000
1   apache02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31    0.000000    0.000000    0.000000    0.000000    0.000000    0.000000
2   apache02                            OSLinux-CPU_CPU_CPULoad     31    0.000000    0.140000    0.020323    0.000000    0.070000    0.080000
3   apache02                         OSLinux-CPU_CPU_CPUSysTime     31    0.228586    0.478100    0.342289    0.340700    0.465600    0.471766
4   apache02                        OSLinux-CPU_CPU_CPUUserTime     31    0.128900    0.261900    0.185965    0.182853    0.232800    0.241099
5   apache02                             OSLinux-CPU_CPU_CPUWio     31    0.012473    0.332600    0.087952    0.033300    0.207800    0.309650
6   apache02                        OSLinux-CPU_CPU_CPUidleutil     31   99.073000   99.588546   99.363154   99.372300   99.521969   99.567800
7   apache01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     31    0.400000    0.500000    0.496774    0.500000    0.500000    0.500000
8   apache01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     31    4.000000    4.000000    4.000000    4.000000    4.000000    4.000000
9   apache01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31    0.000000    0.000000    0.000000    0.000000    0.000000    0.000000
10  apache01                            OSLinux-CPU_CPU_CPULoad     31    1.000000    1.210000    1.071935    1.060000    1.150000    1.195000
11  apache01                         OSLinux-CPU_CPU_CPUSysTime     31   18.147100   18.545439   18.348365   18.351000   18.484672   18.535993
12  apache01                        OSLinux-CPU_CPU_CPUUserTime     31    7.043482    7.384800    7.208178    7.200698    7.353430    7.371100
13  apache01                             OSLinux-CPU_CPU_CPUWio     31    0.012456    0.174400    0.060692    0.029077    0.149539    0.157800
14  apache01                        OSLinux-CPU_CPU_CPUidleutil     31   74.144500   74.541100   74.353356   74.363400   74.492212   74.527636
15  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...     31  500.000000  500.000000  500.000000  500.000000  500.000000  500.000000
16  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31    0.000000    0.000000    0.000000    0.000000    0.000000    0.000000
17  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31    0.000000    0.000000    0.000000    0.000000    0.000000    0.000000
18  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31    0.000000    0.000000    0.000000    0.000000    0.000000    0.000000
19  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31    0.000000    0.000000    0.000000    0.000000    0.000000    0.000000,      cmdb_id                         trace_name  count          min          max         mean          p50          p90          p95
0   dockerA2               trace.self.row_count     30  1354.000000  3311.000000  2211.566667  2330.000000  2781.000000  3001.950000
1       IG01           trace.self.duration_mean     30     0.356945     0.770811     0.548083     0.552825     0.708739     0.739478
2   dockerB2            trace.to_MG01.row_count     30    52.000000   395.000000   200.233333   193.000000   316.600000   333.200000
3   dockerA2        trace.to_MG01.duration_mean     30     0.006450     0.303016     0.071280     0.069130     0.119129     0.127857
4   dockerB1      trace.from_MG01.duration_mean     30     0.152435     1.129526     0.598526     0.580110     0.825259     0.860144
5   dockerA2            trace.to_MG01.row_count     30    12.000000   111.000000    46.300000    42.000000    72.900000    93.650000
6   dockerA2         trace.to_MG01.duration_p95     30     0.009000     1.931700     0.483210     0.280850     1.012500     1.016025
7   dockerA2      trace.from_MG01.duration_mean     30     0.497514     0.906735     0.626704     0.592568     0.763789     0.786984
8   dockerA2       trace.from_MG01.duration_p95     30     1.851800     3.097750     2.115760     2.082175     2.120105     2.132572
9   dockerA2            trace.self.duration_p95     30     1.014000     1.019000     1.016432     1.016000     1.018000     1.018522
10  dockerA2          trace.from_MG01.row_count     30   106.000000   412.000000   186.300000   164.500000   315.400000   387.300000
11  dockerA2           trace.self.duration_mean     30     0.062974     0.109229     0.084754     0.083648     0.099489     0.103065
12      MG01      trace.from_Tomcat02.row_count     30   177.000000   516.000000   270.400000   239.500000   442.600000   485.600000
13      MG01   trace.from_Tomcat02.duration_p95     30     1.076800     3.198500     1.983248     2.090875     2.317220     2.759420
14      MG01   trace.from_dockerB2.duration_p95     30     0.009000     1.015900     0.511310     0.373750     1.012685     1.014100
15      MG01  trace.from_dockerB1.duration_mean     30     0.009743     0.222149     0.075931     0.070946     0.103694     0.119143
16      MG01      trace.from_dockerA2.row_count     30    12.000000   111.000000    46.300000    42.000000    72.900000    93.650000
17      MG01      trace.from_dockerB1.row_count     30    35.000000   365.000000   196.700000   200.000000   274.300000   300.650000
18      MG01   trace.from_dockerB1.duration_p95     30     0.009300     1.212400     0.602898     0.541400     1.013020     1.014080
19      MG01  trace.from_dockerB2.duration_mean     30     0.021885     0.163401     0.068874     0.063820     0.103400     0.130091,      cmdb_id          log_name  count    min      max        mean     p50       p90      p95
0   apache02   log.error_count     31   0.00     0.00    0.000000    0.00     0.000     0.00
1   Tomcat03   log.error_count     31   0.00     0.00    0.000000    0.00     0.000     0.00
2   Tomcat01     log.row_count     31   8.00   573.00  440.258065  460.00   494.000   521.50
3   Tomcat01   log.error_count     31   0.00     0.00    0.000000    0.00     0.000     0.00
4   apache02     log.row_count     31  16.00  1092.00  880.129032  931.00  1019.000  1037.00
5   apache01     log.row_count     31  17.00  1090.00  888.548387  921.00   998.000  1032.50
6   Tomcat04   log.error_count     31   0.00     0.00    0.000000    0.00     0.000     0.00
7   Tomcat03     log.row_count     31  10.00   588.00  443.903226  460.00   514.000   552.50
8   apache01   log.error_count     31   0.00     0.00    0.000000    0.00     0.000     0.00
9   Tomcat04     log.row_count     31   9.00   536.00  438.516129  454.00   504.000   510.50
10   Mysql01     log.row_count     28   1.00    20.00    7.642857    7.00    13.000    16.25
11   Mysql01   log.error_count     28   0.00     0.00    0.000000    0.00     0.000     0.00
12  Tomcat01  log.gc_pause_max     24   0.03     0.12    0.077500    0.08     0.097     0.10
13  Tomcat03  log.gc_pause_max     23   0.03     0.21    0.068696    0.07     0.088     0.09
14  Tomcat04  log.gc_pause_max     23   0.06     0.15    0.086522    0.08     0.120     0.12
15  Tomcat02  log.gc_pause_max     21   0.03     0.23    0.092857    0.08     0.130     0.17
16  Tomcat02   log.error_count     21   0.00     0.00    0.000000    0.00     0.000     0.00
17  Tomcat02     log.row_count     21   1.00     2.00    1.047619    1.00     1.000     1.00
18   Mysql02     log.row_count      1   1.00     1.00    1.000000    1.00     1.000     1.00
19   Mysql02   log.error_count      1   0.00     0.00    0.000000    0.00     0.000     0.00, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [cmdb_id, timestamp, dt, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# Reuse existing df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
    df_metric['dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

# Ensure dt exists and is timezone-aware Asia/Shanghai
tz = pytz.timezone('Asia/Shanghai')
if 'dt' not in df_metric.columns:
    df_metric['dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # convert to tz if not already
    if df_metric['dt'].dt.tz is None:
        df_metric['dt'] = df_metric['dt'].dt.tz_localize(pytz.utc).dt.tz_convert(tz)
    else:
        df_metric['dt'] = df_metric['dt'].dt.tz_convert(tz)

# 1) Compute global P95 threshold per (cmdb_id, kpi_name) using full series (no filtering)
global_p95 = df_metric.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).reset_index().rename(columns={'value': 'global_P95'})

# 2) Filter metric.csv to the incident time window: March 9, 2021, 14:00 to 14:30 (inclusive) in UTC+8
start = pd.Timestamp('2021-03-09 14:00:00', tz=tz)
end   = pd.Timestamp('2021-03-09 14:30:00', tz=tz)
df_win = df_metric[(df_metric['dt'] >= start) & (df_metric['dt'] <= end)].copy()

# If no points in window, prepare empty result
if df_win.empty:
    metric_anom_summary = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','global_P95','anomaly_count_within_window',
        'earliest_anomaly_timestamp','max_value_within_window','total_points_in_window'
    ])
else:
    # 3) Attach global_P95 to windowed points
    df_win = df_win.merge(global_p95, on=['cmdb_id','kpi_name'], how='left')

    # mark anomalies where value >= global_P95 (note: global_P95 is computed from full series as required)
    df_win['is_anomaly'] = df_win['value'] >= df_win['global_P95']

    # aggregate per combo for windowed points
    agg_window = df_win.groupby(['cmdb_id','kpi_name']).agg(
        total_points_in_window = ('value','count'),
        anomaly_count_within_window = ('is_anomaly','sum'),
        max_value_within_window = ('value','max'),
        earliest_anomaly_timestamp = ('timestamp', lambda s: int(s[df_win.loc[s.index,'is_anomaly']].min()) if s[df_win.loc[s.index,'is_anomaly']].any() else pd.NA)
    ).reset_index()

    # Attach global_P95 (unique per combo)
    agg_window = agg_window.merge(global_p95, on=['cmdb_id','kpi_name'], how='left')

    # Keep only combos that have at least one anomaly point in the window
    metric_anom_summary = agg_window[agg_window['anomaly_count_within_window'] > 0].copy()

    # Round numeric columns for compactness
    metric_anom_summary['global_P95'] = metric_anom_summary['global_P95'].round(6)
    metric_anom_summary['max_value_within_window'] = metric_anom_summary['max_value_within_window'].round(6)

    # Ensure earliest_anomaly_timestamp is integer (seconds) or NA
    metric_anom_summary['earliest_anomaly_timestamp'] = metric_anom_summary['earliest_anomaly_timestamp'].where(metric_anom_summary['earliest_anomaly_timestamp'].notna(), pd.NA)

    # Select and order columns as requested
    metric_anom_summary = metric_anom_summary[
        ['cmdb_id','kpi_name','global_P95','anomaly_count_within_window',
         'earliest_anomaly_timestamp','max_value_within_window','total_points_in_window']
    ]

    # 4) Sort by anomaly_count_within_window descending and return top 20
    metric_anom_summary = metric_anom_summary.sort_values(
        by=['anomaly_count_within_window','max_value_within_window'],
        ascending=[False, False]
    ).reset_index(drop=True).head(20)

# Final compact output variable
metric_anom_summary

```
Out[2]:
```
import pandas as pd
import pytz

# Reuse metric_anom_summary from the kernel
try:
    metric_anom_summary
except NameError:
    metric_anom_summary = pd.DataFrame()

tz = pytz.timezone('Asia/Shanghai')

if metric_anom_summary.empty:
    summary = "No metric anomalies were found in the March 9, 2021 14:00–14:30 window."
else:
    n = len(metric_anom_summary)
    # Check if anomaly counts are uniform
    unique_counts = metric_anom_summary['anomaly_count_within_window'].unique()
    uniform_count = unique_counts[0] if len(unique_counts) == 1 else None
    earliest_ts = int(metric_anom_summary['earliest_anomaly_timestamp'].min())
    earliest_dt = pd.to_datetime(earliest_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    examples = []
    for _, r in metric_anom_summary.head(6).iterrows():
        examples.append(
            f"{r['cmdb_id']} | {r['kpi_name']} | P95={r['global_P95']:.6g} | "
            f"anomalies={int(r['anomaly_count_within_window'])} | max_in_window={r['max_value_within_window']:.6g}"
        )
    summary = (
        f"Metric anomaly summary for March 9, 2021 14:00–14:30 (UTC+8):\n\n"
        f"- Found {n} (cmdb_id, kpi_name) combos with values >= their global P95 within the window.\n"
        f"- Earliest anomaly observed at {earliest_dt} (timestamp {earliest_ts}).\n"
        + (f"- Each reported combo had {uniform_count} anomaly points in the window.\n" if uniform_count is not None else "")
        f"- Top examples (up to 6):\n  - " + "\n  - ".join(examples) +
        "\n\nAll reported combos had at least one point (many had all sampled points) in the 14:00–14:30 window exceeding their global P95 thresholds."
    )

summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name    global_P95  anomaly_count_within_window  earliest_anomaly_timestamp  max_value_within_window  total_points_in_window
0           IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09                           31                  1615269600             2.040136e+09                      31
1           IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09                           31                  1615269600             2.040136e+09                      31
2           MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09                           31                  1615269600             2.040136e+09                      31
3           MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09                           31                  1615269600             2.040136e+09                      31
4           MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  9.471166e+08                           31                  1615269600             9.471166e+08                      31
5           MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  5.069049e+08                           31                  1615269600             5.069049e+08                      31
6       Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  4.053960e+05                           31                  1615269600             4.053960e+05                      31
7       Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  3.303510e+05                           31                  1615269600             3.303510e+05                      31
8       Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  3.300540e+05                           31                  1615269600             3.300540e+05                      31
9       Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  3.248520e+05                           31                  1615269600             3.248520e+05                      31
10      Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  9.250000e+02                           31                  1615269600             9.250000e+02                      31
11      Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  5.000000e+02                           31                  1615269600             5.000000e+02                      31
12      Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  5.000000e+02                           31                  1615269600             5.000000e+02                      31
13      Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  5.000000e+02                           31                  1615269600             5.000000e+02                      31
14      Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  5.000000e+02                           31                  1615269600             5.000000e+02                      31
15      Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  2.960000e+02                           31                  1615269600             2.960000e+02                      31
16      Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  2.700000e+02                           31                  1615269600             2.700000e+02                      31
17      Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  2.290000e+02                           31                  1615269600             2.290000e+02                      31
18  ServiceTest1                                             app.rr  1.000000e+02                           31                  1615269600             1.000000e+02                      31
19  ServiceTest1                                             app.sr  1.000000e+02                           31                  1615269600             1.000000e+02                      31```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing df_metric if available; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure timezone-aware datetime column in UTC+8
tz = pytz.timezone('Asia/Shanghai')
if 'dt' not in df_metric.columns:
    df_metric['dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure tz-aware and converted to Asia/Shanghai
    if df_metric['dt'].dt.tz is None:
        df_metric['dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        df_metric['dt'] = df_metric['dt'].dt.tz_convert(tz)

# 1) Compute global P95 and global P5 per (cmdb_id, kpi_name) using full series
q = df_metric.groupby(['cmdb_id', 'kpi_name'])['value'].quantile
global_p95 = q(0.95).reset_index().rename(columns={'value': 'global_P95'})
global_p5  = q(0.05).reset_index().rename(columns={'value': 'global_P5'})

global_thresholds = global_p95.merge(global_p5, on=['cmdb_id','kpi_name'], how='left')

# 2) Filter metric.csv to the incident time window: March 9, 2021 14:00 to 14:30 (inclusive) UTC+8
start = pd.Timestamp('2021-03-09 14:00:00', tz=tz)
end   = pd.Timestamp('2021-03-09 14:30:00', tz=tz)
df_win = df_metric[(df_metric['dt'] >= start) & (df_metric['dt'] <= end)].copy()

# If window is empty, return empty structured DataFrame
cols_out = [
    'cmdb_id','kpi_name','global_P95','global_P5',
    'count_ge_p95','count_le_p5','earliest_ge_p95_timestamp','earliest_le_p5_timestamp',
    'max_value_in_window','min_value_in_window','total_points_in_window'
]
if df_win.empty:
    metric_window_summary = pd.DataFrame(columns=cols_out)
else:
    # 3) Attach global thresholds to windowed points (thresholds computed from full series)
    df_win = df_win.merge(global_thresholds, on=['cmdb_id','kpi_name'], how='left')

    # mark points relative to thresholds
    df_win['is_ge_p95'] = df_win['value'] >= df_win['global_P95']
    df_win['is_le_p5']  = df_win['value'] <= df_win['global_P5']

    # aggregate counts, min/max, totals
    agg_basic = df_win.groupby(['cmdb_id','kpi_name']).agg(
        total_points_in_window = ('value','count'),
        count_ge_p95 = ('is_ge_p95','sum'),
        count_le_p5  = ('is_le_p5','sum'),
        max_value_in_window = ('value','max'),
        min_value_in_window = ('value','min')
    ).reset_index()

    # earliest timestamps for each condition (timestamp is in seconds)
    earliest_ge = df_win[df_win['is_ge_p95']].groupby(['cmdb_id','kpi_name'])['timestamp'].min().reset_index().rename(columns={'timestamp':'earliest_ge_p95_timestamp'})
    earliest_le = df_win[df_win['is_le_p5']].groupby(['cmdb_id','kpi_name'])['timestamp'].min().reset_index().rename(columns={'timestamp':'earliest_le_p5_timestamp'})

    # merge all together
    metric_window_summary = agg_basic.merge(global_thresholds, on=['cmdb_id','kpi_name'], how='left')
    metric_window_summary = metric_window_summary.merge(earliest_ge, on=['cmdb_id','kpi_name'], how='left')
    metric_window_summary = metric_window_summary.merge(earliest_le, on=['cmdb_id','kpi_name'], how='left')

    # 4) Exclude KPI names that are clearly static/config (case-insensitive substring match)
    exclude_pattern = r'HeapMemoryMax|MaxThreads|ErrorCount'
    mask_exclude = metric_window_summary['kpi_name'].str.contains(exclude_pattern, case=False, na=False)
    metric_window_summary = metric_window_summary[~mask_exclude].copy()

    # 5) Keep only combos where either count_ge_p95>0 or count_le_p5>0
    metric_window_summary = metric_window_summary[
        (metric_window_summary['count_ge_p95'] > 0) | (metric_window_summary['count_le_p5'] > 0)
    ].copy()

    if not metric_window_summary.empty:
        # sort by max(count_ge_p95, count_le_p5) descending, then by total_points_in_window desc, then max value desc
        metric_window_summary['primary_count'] = metric_window_summary[['count_ge_p95','count_le_p5']].max(axis=1)
        metric_window_summary = metric_window_summary.sort_values(
            by=['primary_count','total_points_in_window','max_value_in_window'],
            ascending=[False,False,False]
        ).drop(columns=['primary_count'])

        # Select and format columns as requested, round numeric thresholds and values for compactness
        metric_window_summary['global_P95'] = metric_window_summary['global_P95'].round(6)
        metric_window_summary['global_P5']  = metric_window_summary['global_P5'].round(6)
        metric_window_summary['max_value_in_window'] = metric_window_summary['max_value_in_window'].round(6)
        metric_window_summary['min_value_in_window'] = metric_window_summary['min_value_in_window'].round(6)

        metric_window_summary = metric_window_summary[
            ['cmdb_id','kpi_name','global_P95','global_P5',
             'count_ge_p95','count_le_p5','earliest_ge_p95_timestamp','earliest_le_p5_timestamp',
             'max_value_in_window','min_value_in_window','total_points_in_window']
        ].reset_index(drop=True)

        metric_window_summary = metric_window_summary.head(20)
    else:
        # keep empty with columns
        metric_window_summary = metric_window_summary[cols_out]

# Final compact output
metric_window_summary

```
Out[3]:
```
Summary (metric.csv, March 9 2021 14:00–14:30 UTC+8):

- Returned combos: 20 (top matches where at least one point in the window met P95 or P5). Static/config KPIs with substrings HeapMemoryMax, MaxThreads, ErrorCount were excluded per request.

- Key finding: most returned KPIs are effectively constant in the full series (global P95 == global P5 == observed value). As a result, all points in the 14:00–14:30 window meet both >=P95 and <=P5 for those KPIs. Earliest anomaly timestamps are 1615269600 (2021-03-09 14:00:00 UTC+8).

- Top examples (compact):
  - MG02 | JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed: global_P95=947,116,648; count_ge_p95=31; count_le_p5=31; earliest=1615269600; max=947,116,648; total_points=31
  - MG01 | JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed: global_P95=506,904,928; count_ge_p95=31; count_le_p5=31; earliest=1615269600; max=506,904,928; total_points=31
  - Tomcat01 | Tomcat-Requests...MaxTimeRemote (truncated name): global_P95=405,396; count_ge_p95=31; count_le_p5=31; earliest=1615269600; max=405,396; total_points=31
  - Tomcat03 / Tomcat02 / Tomcat04 have similar entries (global_P95 ≈ 330k, 330k, 324k respectively) with 31/31 counts.
  - Multiple ServiceTest* entries (app.rr, app.sr) show global_P95=100 and count 31/31.
  - One exception: ServiceTest6 app.rr has count_ge_p95=30 (one point not >=P95) but count_le_p5=31; total_points=31.

- Interpretation & next steps:
  - The identical P5/P95 values and full-window hits indicate zero or near-zero variance in these KPI series (constant readings). This can be caused by static/config metrics, synthetic/test services, or mis-normalized data.
  - Recommend inspecting raw time series for these KPIs (a few sample points) to confirm they are truly constant and to verify whether using P5/P95 thresholds is meaningful for them.
  - If you want, I can extract small raw samples (timestamps & values) for any specific combo to validate.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name   global_P95    global_P5  count_ge_p95  count_le_p5  earliest_ge_p95_timestamp  earliest_le_p5_timestamp  max_value_in_window  min_value_in_window  total_points_in_window
0            MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  947116648.0  947116648.0            31           31                 1615269600                1615269600          947116648.0         9.471166e+08                      31
1            MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  506904928.0  506904928.0            31           31                 1615269600                1615269600          506904928.0         5.069049e+08                      31
2        Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...     405396.0     405396.0            31           31                 1615269600                1615269600             405396.0         4.053960e+05                      31
3        Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...     330351.0     330351.0            31           31                 1615269600                1615269600             330351.0         3.303510e+05                      31
4        Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...     330054.0     330054.0            31           31                 1615269600                1615269600             330054.0         3.300540e+05                      31
5        Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...     324852.0     324852.0            31           31                 1615269600                1615269600             324852.0         3.248520e+05                      31
6    ServiceTest1                                             app.rr        100.0        100.0            31           31                 1615269600                1615269600                100.0         1.000000e+02                      31
7    ServiceTest1                                             app.sr        100.0        100.0            31           31                 1615269600                1615269600                100.0         1.000000e+02                      31
8   ServiceTest10                                             app.rr        100.0        100.0            31           31                 1615269600                1615269600                100.0         1.000000e+02                      31
9   ServiceTest10                                             app.sr        100.0        100.0            31           31                 1615269600                1615269600                100.0         1.000000e+02                      31
10  ServiceTest11                                             app.rr        100.0        100.0            31           31                 1615269600                1615269600                100.0         1.000000e+02                      31
11  ServiceTest11                                             app.sr        100.0        100.0            31           31                 1615269600                1615269600                100.0         1.000000e+02                      31
12   ServiceTest2                                             app.rr        100.0        100.0            31           31                 1615269600                1615269600                100.0         1.000000e+02                      31
13   ServiceTest2                                             app.sr        100.0        100.0            31           31                 1615269600                1615269600                100.0         1.000000e+02                      31
14   ServiceTest3                                             app.rr        100.0        100.0            31           31                 1615269600                1615269600                100.0         1.000000e+02                      31
15   ServiceTest3                                             app.sr        100.0        100.0            31           31                 1615269600                1615269600                100.0         1.000000e+02                      31
16   ServiceTest4                                             app.sr        100.0        100.0            31           31                 1615269600                1615269600                100.0         1.000000e+02                      31
17   ServiceTest5                                             app.rr        100.0        100.0            31           31                 1615269600                1615269600                100.0         1.000000e+02                      31
18   ServiceTest5                                             app.sr        100.0        100.0            31           31                 1615269600                1615269600                100.0         1.000000e+02                      31
19   ServiceTest6                                             app.rr        100.0        100.0            30           31                 1615269600                1615269600                100.0         9.917000e+01                      31```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# Reuse df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure timezone-aware datetime column in UTC+8
tz = pytz.timezone('Asia/Shanghai')
if 'dt' not in df_metric.columns:
    df_metric['dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure tz-aware and converted to Asia/Shanghai
    try:
        if df_metric['dt'].dt.tz is None:
            df_metric['dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        else:
            df_metric['dt'] = df_metric['dt'].dt.tz_convert(tz)
    except Exception:
        df_metric['dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 and global P5 and full-series min/max per (cmdb_id, kpi_name)
grp = df_metric.groupby(['cmdb_id','kpi_name'])['value']
global_stats = grp.agg(
    global_P95 = lambda x: x.quantile(0.95),
    global_P5  = lambda x: x.quantile(0.05),
    full_min   = 'min',
    full_max   = 'max'
).reset_index()

# 2) Mark variable series where min != max
global_stats['is_variable'] = global_stats['full_min'] != global_stats['full_max']

# total number of variable series
variable_count = int(global_stats['is_variable'].sum())

# 3) Filter to only variable series
variable_thresholds = global_stats[global_stats['is_variable']].copy()

# 4) Filter metric points to the incident window March 9, 2021 14:00 to 14:30 (inclusive) UTC+8
start = pd.Timestamp('2021-03-09 14:00:00', tz=tz)
end   = pd.Timestamp('2021-03-09 14:30:00', tz=tz)
df_win = df_metric[(df_metric['dt'] >= start) & (df_metric['dt'] <= end)].copy()

# Keep only window points that belong to variable series
if df_win.empty or variable_thresholds.empty:
    metric_window_variable_summary = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','global_P95','global_P5',
        'count_ge_p95','count_le_p5','earliest_ge_p95_timestamp','earliest_le_p5_timestamp',
        'max_value_in_window','min_value_in_window','total_points_in_window'
    ])
else:
    # merge thresholds for variable series into window points
    df_win = df_win.merge(
        variable_thresholds[['cmdb_id','kpi_name','global_P95','global_P5']],
        on=['cmdb_id','kpi_name'],
        how='inner'
    )

    # If no variable combos present in window, produce empty result
    if df_win.empty:
        metric_window_variable_summary = pd.DataFrame(columns=[
            'cmdb_id','kpi_name','global_P95','global_P5',
            'count_ge_p95','count_le_p5','earliest_ge_p95_timestamp','earliest_le_p5_timestamp',
            'max_value_in_window','min_value_in_window','total_points_in_window'
        ])
    else:
        # 5) Compute required per-combo aggregations
        df_win['is_ge_p95'] = df_win['value'] >= df_win['global_P95']
        df_win['is_le_p5']  = df_win['value'] <= df_win['global_P5']

        agg_basic = df_win.groupby(['cmdb_id','kpi_name']).agg(
            total_points_in_window = ('value','count'),
            count_ge_p95 = ('is_ge_p95','sum'),
            count_le_p5  = ('is_le_p5','sum'),
            max_value_in_window = ('value','max'),
            min_value_in_window = ('value','min')
        ).reset_index()

        earliest_ge = df_win[df_win['is_ge_p95']].groupby(['cmdb_id','kpi_name'])['timestamp'].min().reset_index().rename(columns={'timestamp':'earliest_ge_p95_timestamp'})
        earliest_le = df_win[df_win['is_le_p5']].groupby(['cmdb_id','kpi_name'])['timestamp'].min().reset_index().rename(columns={'timestamp':'earliest_le_p5_timestamp'})

        metric_window_variable_summary = agg_basic.merge(
            variable_thresholds[['cmdb_id','kpi_name','global_P95','global_P5']],
            on=['cmdb_id','kpi_name'], how='left'
        ).merge(earliest_ge, on=['cmdb_id','kpi_name'], how='left').merge(earliest_le, on=['cmdb_id','kpi_name'], how='left')

        # 6) Keep only combos where count_ge_p95>0 or count_le_p5>0
        metric_window_variable_summary = metric_window_variable_summary[
            (metric_window_variable_summary['count_ge_p95'] > 0) | (metric_window_variable_summary['count_le_p5'] > 0)
        ].copy()

        if not metric_window_variable_summary.empty:
            # sort by max(count_ge_p95, count_le_p5) desc
            metric_window_variable_summary['primary_count'] = metric_window_variable_summary[['count_ge_p95','count_le_p5']].max(axis=1)
            metric_window_variable_summary = metric_window_variable_summary.sort_values(
                by=['primary_count','total_points_in_window','max_value_in_window'],
                ascending=[False,False,False]
            ).drop(columns=['primary_count'])

            # format numeric columns compactly
            metric_window_variable_summary['global_P95'] = metric_window_variable_summary['global_P95'].round(6)
            metric_window_variable_summary['global_P5']  = metric_window_variable_summary['global_P5'].round(6)
            metric_window_variable_summary['max_value_in_window'] = metric_window_variable_summary['max_value_in_window'].round(6)
            metric_window_variable_summary['min_value_in_window'] = metric_window_variable_summary['min_value_in_window'].round(6)

            # select requested columns only and limit to top 20
            metric_window_variable_summary = metric_window_variable_summary[
                ['cmdb_id','kpi_name','global_P95','global_P5',
                 'count_ge_p95','count_le_p5','earliest_ge_p95_timestamp','earliest_le_p5_timestamp',
                 'max_value_in_window','min_value_in_window','total_points_in_window']
            ].reset_index(drop=True).head(20)
        else:
            # empty with columns
            metric_window_variable_summary = metric_window_variable_summary[[
                'cmdb_id','kpi_name','global_P95','global_P5',
                'count_ge_p95','count_le_p5','earliest_ge_p95_timestamp','earliest_le_p5_timestamp',
                'max_value_in_window','min_value_in_window','total_points_in_window'
            ]]

# Final outputs: metric_window_variable_summary (compact table) and variable_count (total number of variable series)
metric_window_variable_summary, variable_count

```
Out[4]:
```
Summary (atomic results from metric.csv, variable series only, window 2021-03-09 14:00–14:30 UTC+8):

- Total number of variable (cmdb_id, kpi_name) series in the full dataset: 849.

- Returned up to 20 variable series that had values in the window meeting >= global P95 or <= global P5. All earliest anomaly timestamps are within the window; many earliest events occur at 2021-03-09 14:00:00 (timestamp 1615269600, UTC+8).

- Top example rows (compact):
  1. ServiceTest6 | app.rr — global_P95=100.0, global_P5=100.00, count_ge_p95=30, count_le_p5=31, earliest_ge=2021-03-09 14:00:00, earliest_le=2021-03-09 14:00:00, max=100.0, min=99.17, total_points=31
  2. ServiceTest7 | app.rr — global_P95=100.0, global_P5=100.00, count_ge_p95=30, count_le_p5=31, earliest_ge=2021-03-09 14:00:00, earliest_le=2021-03-09 14:00:00, max=100.0, min=99.36, total_points=31
  3. Mysql02 | OSLinux-...TCP-FIN-WAIT — global_P95=0.0, global_P5=0.00, count_ge_p95=31, count_le_p5=30, earliest_ge=2021-03-09 14:00:00, earliest_le=2021-03-09 14:00:00, max=1.0, min=0.0, total_points=31
  4. apache01 | OSLinux-...zabbix_agentd_PROC... — global_P95=0.5, global_P5=0.50, count_ge_p95=30, count_le_p5=31, earliest_ge=2021-03-09 14:00:00, earliest_le=2021-03-09 14:00:00, max=0.5, min=0.4, total_points=31
  5. Mysql01 | OSLinux-...zabbix_agentd_PROC... — global_P95=0.4, global_P5=0.40, count_ge_p95=30, count_le_p5=31, earliest_ge=2021-03-09 14:00:00, earliest_le=2021-03-09 14:00:00, max=0.4, min=0.3, total_points=31
  6. Redis02 | OSLinux-...zabbix_agentd_PROC... — global_P95=0.4, global_P5=0.40, count_ge_p95=30, count_le_p5=31, earliest_ge=2021-03-09 14:00:00, earliest_le=2021-03-09 14:00:00, max=0.4, min=0.3, total_points=31
  7. IG01 | JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed — global_P95≈57,914,504, global_P5≈57,912,064, count_ge_p95=26, count_le_p5=5, earliest_ge=2021-03-09 14:05:00 (1615269900), earliest_le=2021-03-09 14:00:00, max≈57,914,504, min≈57,912,064, total_points=31
  8. ServiceTest4 | app.rr — global_P95=100.0, global_P5=98.55, count_ge_p95=26, count_le_p5=2, earliest_ge=2021-03-09 14:00:00, earliest_le=2021-03-09 14:13:00 (1615270380), max=100.0, min=96.13, total_points=31
  9. Mysql02 | OSLinux-...MEMUsedMemPerc — global_P95=98.0, global_P5=98.00, count_ge_p95=20, count_le_p5=21, earliest_ge=2021-03-09 14:01:00 (1615269660), earliest_le=2021-03-09 14:01:00, max=98.0, min=97.0, total_points=21
 10. Tomcat04 | OSLinux-...MEMUsedMemPerc — global_P95=98.0, global_P5=97.00, count_ge_p95=19, count_le_p5=3, earliest_ge=2021-03-09 14:00:00, earliest_le=2021-03-09 14:10:00 (1615270200), max=98.0, min=97.0, total_points=22

- Notes / interpretation:
  - Many series have P5 and P95 values equal or very close, so window points can simultaneously satisfy <=P5 and >=P95 (indicating low variance or narrow distribution).
  - Earliest anomaly timestamps mostly occur at 14:00:00 (start of the window), with some earliest_ge or earliest_le at later minutes (e.g., 14:01, 14:03, 14:05, 14:10, 14:13) as shown above.
  - The result list is limited to up to 20 variable series that had at least one point beyond the global P5/P95 thresholds; the full count of variable series is 849.

If you want, I can (a) show the full variable-series table, (b) return raw sample points for any specific combo above, or (c) compute alternative thresholds (e.g., P90, mean±std) to reduce false positives for near-constant series.

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id                                           kpi_name    global_P95    global_P5  count_ge_p95  count_le_p5  earliest_ge_p95_timestamp  earliest_le_p5_timestamp  max_value_in_window  min_value_in_window  total_points_in_window
0   ServiceTest6                                             app.rr  1.000000e+02       100.00            30           31                 1615269600                1615269600         1.000000e+02                99.17                      31
1   ServiceTest7                                             app.rr  1.000000e+02       100.00            30           31                 1615269600                1615269600         1.000000e+02                99.36                      31
2        Mysql02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  0.000000e+00         0.00            31           30                 1615269600                1615269600         1.000000e+00                 0.00                      31
3       apache01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  5.000000e-01         0.50            30           31                 1615269600                1615269600         5.000000e-01                 0.40                      31
4        Mysql01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  4.000000e-01         0.40            30           31                 1615269600                1615269600         4.000000e-01                 0.30                      31
5        Redis02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  4.000000e-01         0.40            30           31                 1615269600                1615269600         4.000000e-01                 0.30                      31
6           IG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  3.000000e-01         0.30            30           31                 1615269600                1615269600         3.000000e-01                 0.20                      31
7           IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.791450e+07  57912064.00            26            5                 1615269900                1615269600         5.791450e+07          57912064.00                      31
8   ServiceTest4                                             app.rr  1.000000e+02        98.55            26            2                 1615269600                1615270380         1.000000e+02                96.13                      31
9        Mysql02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  6.000000e-01         0.50            23            8                 1615269600                1615269780         6.000000e-01                 0.50                      31
10       Mysql01         Mysql-MySQL_3306_max trx lock memory bytes  0.000000e+00         0.00            21           20                 1615269600                1615269600         1.136000e+03                 0.00                      21
11       Mysql02       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc  9.800000e+01        98.00            20           21                 1615269660                1615269660         9.800000e+01                97.00                      21
12       Mysql01               Mysql-MySQL_3306_Max trx rows locked  0.000000e+00         0.00            21           20                 1615269600                1615269600         1.000000e+00                 0.00                      21
13       Mysql01                Mysql-MySQL_3306_MaxTrxRowsModified  0.000000e+00         0.00            21           20                 1615269600                1615269600         1.000000e+00                 0.00                      21
14       Mysql01                     Mysql-MySQL_3306_Opened Tables  0.000000e+00         0.00            21           20                 1615269600                1615269660         1.670000e-02                 0.00                      21
15       Mysql01           Mysql-MySQL_3306_Table open cache misses  0.000000e+00         0.00            21           20                 1615269600                1615269660         1.670000e-02                 0.00                      21
16      Tomcat04       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc  9.800000e+01        97.00            19            3                 1615269600                1615270200         9.800000e+01                97.00                      22
17          MG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  1.000000e+00         0.00             2           19                 1615269900                1615269660         1.000000e+00                 0.00                      21
18      Tomcat04     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  1.000000e+00         0.00             2           19                 1615269780                1615269600         1.000000e+00                 0.00                      21
19       Mysql01                        Mysql-MySQL_3306_Com Insert  2.667000e-01         0.00             2           19                 1615269600                1615269660         2.667000e-01                 0.00                      21, 849)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz
from itertools import groupby

# Reuse existing df_metric if present
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure timezone-aware datetime column in UTC+8
tz = pytz.timezone('Asia/Shanghai')
if 'dt' not in df_metric.columns:
    df_metric['dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure tz-aware and converted to Asia/Shanghai
    try:
        if df_metric['dt'].dt.tz is None:
            df_metric['dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        else:
            df_metric['dt'] = df_metric['dt'].dt.tz_convert(tz)
    except Exception:
        df_metric['dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 and P5 and full-series min/max per (cmdb_id, kpi_name)
grp = df_metric.groupby(['cmdb_id','kpi_name'])['value']
global_stats = grp.agg(
    global_P95 = lambda x: x.quantile(0.95),
    global_P5  = lambda x: x.quantile(0.05),
    full_min   = 'min',
    full_max   = 'max'
).reset_index()

# 2) Identify variable series (min != max)
variable_stats = global_stats[global_stats['full_min'] != global_stats['full_max']].copy()

# 3) Filter metric rows to the incident window March 9, 2021 14:00 to 14:30 (inclusive) UTC+8
start = pd.Timestamp('2021-03-09 14:00:00', tz=tz)
end   = pd.Timestamp('2021-03-09 14:30:00', tz=tz)
df_win = df_metric[(df_metric['dt'] >= start) & (df_metric['dt'] <= end)].copy()

# Keep only variable series rows in window by merging
df_win = df_win.merge(variable_stats[['cmdb_id','kpi_name','global_P95','global_P5']], on=['cmdb_id','kpi_name'], how='inner')

# If no window rows or no variable series present, return empty results
if df_win.empty:
    df_faults = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','fault_start_timestamp','fault_end_timestamp',
        'fault_length_minutes','max_value_in_fault','min_value_in_fault',
        'global_P95','global_P5','total_points_in_window'
    ])
else:
    # 4) For each variable combo present in the window compute counts to pick top 20 combos
    df_win['is_ge_p95'] = df_win['value'] >= df_win['global_P95']
    df_win['is_le_p5']  = df_win['value'] <= df_win['global_P5']
    agg_counts = df_win.groupby(['cmdb_id','kpi_name']).agg(
        total_points_in_window = ('value','count'),
        count_ge_p95 = ('is_ge_p95','sum'),
        count_le_p5  = ('is_le_p5','sum'),
        global_P95 = ('global_P95','first'),
        global_P5  = ('global_P5','first')
    ).reset_index()

    # compute primary count and select top 20 combos by primary count desc
    agg_counts['primary_count'] = agg_counts[['count_ge_p95','count_le_p5']].max(axis=1)
    top_combos = agg_counts.sort_values(['primary_count','total_points_in_window'], ascending=[False,False]).head(20)
    top_keys = set(zip(top_combos['cmdb_id'], top_combos['kpi_name']))

    # 5) For each selected combo, scan ordered window points to find anomalous consecutive segments
    result_rows = []
    for cmdb_id, kpi_name in top_keys:
        sub = df_win[(df_win['cmdb_id']==cmdb_id) & (df_win['kpi_name']==kpi_name)].copy()
        sub = sub.sort_values('timestamp').reset_index(drop=True)

        # mark anomaly if >=P95 or <=P5
        sub['is_anom'] = (sub['value'] >= sub['global_P95']) | (sub['value'] <= sub['global_P5'])

        # scan for consecutive anomalous segments with minute continuity (60s)
        segments = []
        seg_start_idx = None
        seg_end_idx = None
        for i, row in sub.iterrows():
            if row['is_anom']:
                if seg_start_idx is None:
                    # start new segment
                    seg_start_idx = i
                    seg_end_idx = i
                else:
                    # check continuity: timestamp diff from previous == 60 sec
                    prev_ts = sub.at[i-1,'timestamp']
                    if row['timestamp'] - prev_ts == 60:
                        seg_end_idx = i
                    else:
                        # break in continuity -> close previous segment and start new
                        segments.append((seg_start_idx, seg_end_idx))
                        seg_start_idx = i
                        seg_end_idx = i
            else:
                if seg_start_idx is not None:
                    segments.append((seg_start_idx, seg_end_idx))
                    seg_start_idx = None
                    seg_end_idx = None
        # close last open segment
        if seg_start_idx is not None:
            segments.append((seg_start_idx, seg_end_idx))

        # filter segments with length >=3 minutes (inclusive)
        valid_segments = []
        for s,e in segments:
            length = e - s + 1
            if length >= 3:
                seg_slice = sub.loc[s:e]
                fault_start_ts = int(seg_slice['timestamp'].iloc[0])
                fault_end_ts = int(seg_slice['timestamp'].iloc[-1])
                max_val = seg_slice['value'].max()
                min_val = seg_slice['value'].min()
                valid_segments.append({
                    'fault_start_timestamp': fault_start_ts,
                    'fault_end_timestamp': fault_end_ts,
                    'fault_length_minutes': length,
                    'max_value_in_fault': max_val,
                    'min_value_in_fault': min_val
                })

        # 6) If multiple valid segments, keep only the earliest (lowest start timestamp)
        if valid_segments:
            valid_segments.sort(key=lambda x: x['fault_start_timestamp'])
            seg = valid_segments[0]
            total_points = int(sub.shape[0])
            # fetch global thresholds (same for all rows)
            gp95 = float(sub['global_P95'].iloc[0])
            gp5  = float(sub['global_P5'].iloc[0])
            result_rows.append({
                'cmdb_id': cmdb_id,
                'kpi_name': kpi_name,
                'fault_start_timestamp': seg['fault_start_timestamp'],
                'fault_end_timestamp': seg['fault_end_timestamp'],
                'fault_length_minutes': seg['fault_length_minutes'],
                'max_value_in_fault': seg['max_value_in_fault'],
                'min_value_in_fault': seg['min_value_in_fault'],
                'global_P95': gp95,
                'global_P5': gp5,
                'total_points_in_window': total_points
            })

    # build result DataFrame
    if result_rows:
        df_faults = pd.DataFrame(result_rows)
        # 7) sort by fault_length_minutes desc, then fault_start_timestamp asc, limit 20, and keep specified columns
        df_faults = df_faults.sort_values(['fault_length_minutes','fault_start_timestamp'], ascending=[False,True]).head(20)
        # round numeric columns for compactness
        df_faults['global_P95'] = df_faults['global_P95'].round(6)
        df_faults['global_P5']  = df_faults['global_P5'].round(6)
        df_faults['max_value_in_fault'] = df_faults['max_value_in_fault'].round(6)
        df_faults['min_value_in_fault'] = df_faults['min_value_in_fault'].round(6)
    else:
        df_faults = pd.DataFrame(columns=[
            'cmdb_id','kpi_name','fault_start_timestamp','fault_end_timestamp',
            'fault_length_minutes','max_value_in_fault','min_value_in_fault',
            'global_P95','global_P5','total_points_in_window'
        ])

# Also return total number of variable series found in step 2
variable_series_count = int(variable_stats.shape[0])

# Final compact outputs
df_faults, variable_series_count

```
Out[5]:
```
Summary (atomic fault scan from metric.csv for 2021-03-09 14:00–14:30 UTC+8):

- Total variable series in the dataset (min != max): 849.

- Fault detection results: 14 combos (one earliest fault segment per selected combo) were identified among the top variable combos; a fault is defined as >=3 consecutive anomalous minutes where a point is anomalous if value >= global P95 or value <= global P5.

- Many faults spanned the entire incident window (14:00:00 → 14:30:00, inclusive — 31 consecutive minutes). Examples of full-window faults (fault_length = 31 minutes):
  - apache01 | OSLinux-OSLinux_PROCESS_zabbix-... — values 0.40–0.50, global_P95=0.5, global_P5=0.50
  - Redis02 | OSLinux-OSLinux_PROCESS_zabbix-... — values 0.30–0.40, global_P95=0.4, global_P5=0.40
  - IG01 | OSLinux-OSLinux_PROCESS_zabbix-... — values 0.20–0.30, global_P95=0.3, global_P5=0.30
  - ServiceTest6 | app.rr — values 99.17–100.0, global_P95=100.0, global_P5=100.00
  - Mysql02 | OSLinux-...TCP-FIN-WAIT — values 0.00–1.00, global_P95=0.0, global_P5=0.00
  - Several other combos (Mysql01, ServiceTest7, IG01 JVM memory, etc.) also show full-window anomalous segments.

- Shorter faults found (examples):
  - ServiceTest4 | app.rr — fault from 14:00:00 to 14:18:20? (reported end 1615270200 → 2021-03-09 14:10:00? — note timestamps reported as seconds) — fault_length = 11 minutes, values 100.0.
  - Tomcat04 | MEMORY_MEMUsedMemPerc — fault 14:15:00 → 14:20:00 (6 minutes), values ~98.0.
  - A few combos (Mysql02 MEM, MG01 and Tomcat04 TCP-CLOSE-WAIT) had shortest faults of 3 consecutive minutes.

- Timestamps: earliest anomalies mostly at the window start 2021-03-09 14:00:00 (unix 1615269600). Other fault starts in the window include 14:15, 14:16, 14:18, 14:20, etc., as shown in the results.

- Interpretation and next steps:
  - Many series triggered anomalies continuously across the whole 14:00–14:30 window — this often indicates either a sustained condition during the window or thresholds that coincide with nearly-constant series (narrow distributions).
  - Confirm whether global P5/P95 thresholds are appropriate for these low-variance series (they may cause frequent "anomalies" when series hover near a single value).
  - Recommended follow-ups: inspect raw time-series samples for the listed combos, validate threshold behavior, and correlate with application logs or traces for the same timeframe.

If you want, I can (a) convert all fault timestamps to readable UTC+8 datetimes in a compact list, (b) show the raw point sequences for any specific combo, or (c) adjust the fault rules (e.g., require stricter deviation) and re-run.

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id                                           kpi_name  fault_start_timestamp  fault_end_timestamp  fault_length_minutes  max_value_in_fault  min_value_in_fault  global_P95    global_P5  total_points_in_window
0       apache01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             1615269600           1615271400                    31                 0.5                0.40         0.5         0.50                      31
2        Redis02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             1615269600           1615271400                    31                 0.4                0.30         0.4         0.40                      31
3           IG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             1615269600           1615271400                    31                 0.3                0.20         0.3         0.30                      31
4   ServiceTest6                                             app.rr             1615269600           1615271400                    31               100.0               99.17       100.0       100.00                      31
6        Mysql02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT             1615269600           1615271400                    31                 1.0                0.00         0.0         0.00                      31
7        Mysql01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             1615269600           1615271400                    31                 0.4                0.30         0.4         0.40                      31
8        Mysql02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             1615269600           1615271400                    31                 0.6                0.50         0.6         0.50                      31
10  ServiceTest7                                             app.rr             1615269600           1615271400                    31               100.0               99.36       100.0       100.00                      31
13          IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed             1615269600           1615271400                    31          57914496.0         57912064.00  57914496.0  57912064.00                      31
1   ServiceTest4                                             app.rr             1615269600           1615270200                    11               100.0              100.00       100.0        98.55                      31
9       Tomcat04       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc             1615270500           1615270800                     6                98.0               98.00        98.0        97.00                      22
5        Mysql02       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc             1615270560           1615270680                     3                98.0               98.00        98.0        98.00                      21
11          MG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             1615270560           1615270680                     3                 0.0                0.00         1.0         0.00                      21
12      Tomcat04     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             1615270680           1615270800                     3                 0.0                0.00         1.0         0.00                      21, 849)```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

# Reuse existing df_trace if present; otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# Ensure timezone-aware datetime column in UTC+8
tz = pytz.timezone('Asia/Shanghai')
if 'dt' not in df_trace.columns:
    df_trace['dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure tz-aware and converted to Asia/Shanghai
    try:
        if df_trace['dt'].dt.tz is None:
            df_trace['dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        else:
            df_trace['dt'] = df_trace['dt'].dt.tz_convert(tz)
    except Exception:
        df_trace['dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 and P5 per (cmdb_id, trace_name) using full series
q = df_trace.groupby(['cmdb_id','trace_name'])['value'].quantile
global_p95 = q(0.95).reset_index().rename(columns={'value':'global_P95'})
global_p5  = q(0.05).reset_index().rename(columns={'value':'global_P5'})
trace_thresholds = global_p95.merge(global_p5, on=['cmdb_id','trace_name'], how='left')

# 2) Filter trace.csv to the incident window March 9, 2021 14:00 to 14:30 (inclusive) UTC+8
start = pd.Timestamp('2021-03-09 14:00:00', tz=tz)
end   = pd.Timestamp('2021-03-09 14:30:00', tz=tz)
df_trace_win = df_trace[(df_trace['dt'] >= start) & (df_trace['dt'] <= end)].copy()

# 3) Restrict to given cmdb_id list
cmdb_list = ['IG01','MG01','Tomcat01','Tomcat02','Tomcat03','Tomcat04','Mysql01','Mysql02','apache01','apache02','Redis01','Redis02']
df_trace_win = df_trace_win[df_trace_win['cmdb_id'].isin(cmdb_list)].copy()

# 4) Attach global thresholds (computed from full series) to the windowed rows
df_trace_win = df_trace_win.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')

# If no rows after filtering, prepare empty DataFrame with requested columns
cols_out = [
    'cmdb_id','trace_name','global_P95','global_P5',
    'count_ge_p95','count_le_p5','earliest_ge_p95_timestamp','earliest_le_p5_timestamp',
    'mean_value_in_window','max_value_in_window','total_points_in_window'
]
if df_trace_win.empty:
    trace_anom_summary = pd.DataFrame(columns=cols_out)
else:
    # mark comparisons
    df_trace_win['is_ge_p95'] = df_trace_win['value'] >= df_trace_win['global_P95']
    df_trace_win['is_le_p5']  = df_trace_win['value'] <= df_trace_win['global_P5']

    # aggregate per combo
    agg = df_trace_win.groupby(['cmdb_id','trace_name']).agg(
        total_points_in_window = ('value','count'),
        count_ge_p95 = ('is_ge_p95','sum'),
        count_le_p5  = ('is_le_p5','sum'),
        mean_value_in_window = ('value','mean'),
        max_value_in_window  = ('value','max')
    ).reset_index()

    # earliest timestamps for conditions
    earliest_ge = df_trace_win[df_trace_win['is_ge_p95']].groupby(['cmdb_id','trace_name'])['timestamp'].min().reset_index().rename(columns={'timestamp':'earliest_ge_p95_timestamp'})
    earliest_le = df_trace_win[df_trace_win['is_le_p5']].groupby(['cmdb_id','trace_name'])['timestamp'].min().reset_index().rename(columns={'timestamp':'earliest_le_p5_timestamp'})

    # merge thresholds and earliest times
    summary = agg.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left').merge(earliest_ge, on=['cmdb_id','trace_name'], how='left').merge(earliest_le, on=['cmdb_id','trace_name'], how='left')

    # keep only combos where count_ge_p95>0 or count_le_p5>0
    summary = summary[(summary['count_ge_p95']>0) | (summary['count_le_p5']>0)].copy()

    if summary.empty:
        trace_anom_summary = pd.DataFrame(columns=cols_out)
    else:
        # format numeric cols compactly
        summary['global_P95'] = summary['global_P95'].round(6)
        summary['global_P5']  = summary['global_P5'].round(6)
        summary['mean_value_in_window'] = summary['mean_value_in_window'].round(6)
        summary['max_value_in_window']  = summary['max_value_in_window'].round(6)

        # select and order columns
        trace_anom_summary = summary[
            ['cmdb_id','trace_name','global_P95','global_P5',
             'count_ge_p95','count_le_p5','earliest_ge_p95_timestamp','earliest_le_p5_timestamp',
             'mean_value_in_window','max_value_in_window','total_points_in_window']
        ].copy()

        # sort by max(count_ge_p95,count_le_p5) desc, then limit to top 20
        trace_anom_summary['primary_count'] = trace_anom_summary[['count_ge_p95','count_le_p5']].max(axis=1)
        trace_anom_summary = trace_anom_summary.sort_values(['primary_count'], ascending=[False]).drop(columns=['primary_count']).reset_index(drop=True).head(20)

# Final compact output
trace_anom_summary

```
Out[6]:
```
Summary of trace anomaly scan (window 2021-03-09 14:00–14:30 UTC+8):

What I did
- Computed global P95 and P5 per (cmdb_id, trace_name) using the full series (no window filtering).
- Filtered traces to the incident window and to the specified components (IG01, MG01, Tomcat01–04, Mysql01–02, apache01–02, Redis01–02).
- Reported combos in the window that had points >= global P95 or <= global P5, showing counts, earliest anomaly timestamps, means, maxima, and total points.

Key findings (top examples)
- Tomcat03 — trace.self.duration_p95
  - global_P95 = 1.004, global_P5 = 0.05595
  - count_ge_p95 = 11, count_le_p5 = 2
  - earliest >=P95 at 1615269900 (2021-03-09 14:05:00 UTC+8)
  - mean in window ≈ 0.6003, max = 1.03, total_points = 26
  - This is the most prominent anomaly (many >=P95 points).

- Tomcat02 — trace.self.duration_p95
  - global_P95 = 1.006, count_ge_p95 = 3, count_le_p5 = 2
  - earliest >=P95 at 1615270680 (≈14:18:00)

- MG01 — trace.from_dockerA1.row_count
  - global_P95 = 67.1, global_P5 = 8.0
  - count_ge_p95 = 2, count_le_p5 = 3
  - mean ≈ 36.19, max = 74, total_points = 27

- Tomcat02 — trace.from_IG02.row_count
  - global_P95 = 269.85, count_ge_p95 = 2, count_le_p5 = 3
  - mean ≈ 233.5, max = 283, total_points = 30

- Tomcat04 — several entries (to/from MG01 and IG02)
  - Examples: trace.to_MG01.row_count (count_ge_p95 = 2), trace.from_IG02.duration_mean (count_ge_p95 = 2), duration_p95 (count_ge_p95 = 2)
  - Some earliest >=P95 timestamps around 14:17–14:23.

- IG01 — multiple trace types (self.duration_p95, self.duration_mean, to_Tomcat* duration_mean/p95/row_count)
  - Mostly count_ge_p95 = 2 and count_le_p5 = 2 for these traces; earliest anomaly times vary across the window (e.g., 14:12–14:23).

Overall pattern and interpretation
- Tomcat03.trace.self.duration_p95 is the clearest hotspot (11 points >= global P95 within the window), starting around 14:05.
- Many other trace features show low but nonzero anomaly counts (1–3 points >=P95 or <=P5) — most are IG01 and Tomcat-related paths and some MG01 / Tomcat04 row_count/duration metrics.
- Earliest anomaly timestamps are inside the window (examples: 14:05, 14:12, 14:17–14:23, etc.), not outside it.

Recommended next steps
- Inspect Tomcat03 trace.self.duration traces in detail (raw per-minute values around 14:00–14:30) because it shows sustained P95 exceedances.
- For IG01 and Tomcat02/04 traces with smaller anomaly counts, check whether anomalies are isolated spikes or correlated across services (trace paths show upstream/downstream relationships).
- Correlate these trace anomalies with metrics and logs for the same hosts/times (CPU, GC, request rates, error logs) to identify possible root causes (e.g., slow requests, downstream slowness, resource contention).

If you want, I can:
- Convert all reported earliest timestamps to human-readable datetimes (UTC+8) in a compact table,
- Show the raw per-minute trace points for Tomcat03 and IG01 for 14:00–14:30, or
- Expand the list beyond the top 20 combos.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                       trace_name   global_P95    global_P5  count_ge_p95  count_le_p5  earliest_ge_p95_timestamp  earliest_le_p5_timestamp  mean_value_in_window  max_value_in_window  total_points_in_window
0   Tomcat03          trace.self.duration_p95     1.004000     0.055950            11            2                 1615269900                1615269600              0.600304             1.030000                      26
1   Tomcat02          trace.self.duration_p95     1.006000     0.049168             3            2                 1615270680                1615269600              0.620338             1.013000                      30
2       MG01    trace.from_dockerA1.row_count    67.100000     8.000000             2            3                 1615270680                1615270200             36.185185            74.000000                      27
3   Tomcat02        trace.from_IG02.row_count   269.850000   191.000000             2            3                 1615270380                1615269600            233.500000           283.000000                      30
4   Tomcat04          trace.to_MG01.row_count   425.100000   135.350000             2            2                 1615270620                1615270080            233.125000           495.000000                      24
5   Tomcat04    trace.from_IG02.duration_mean     0.713016     0.331597             2            2                 1615270980                1615270140              0.522060             0.814118                      24
6   Tomcat04     trace.from_IG02.duration_p95     3.106178     1.198960             2            2                 1615270860                1615270020              2.194469             3.152600                      24
7       IG01         trace.self.duration_mean     0.739478     0.377492             2            2                 1615270800                1615269600              0.548083             0.770811                      30
8       IG01      trace.to_Tomcat01.row_count   270.700000   191.150000             2            2                 1615270320                1615269840            223.766667           289.000000                      30
9       IG01          trace.self.duration_p95     3.293187     2.057180             2            2                 1615270440                1615269600              2.413963             4.008000                      30
10      IG01  trace.to_Tomcat01.duration_mean     0.720352     0.351083             2            2                 1615270980                1615269600              0.534646             0.944131                      30
11      IG01             trace.self.row_count  2077.200000  1530.200000             2            2                 1615270320                1615270080           1805.533333          2252.000000                      30
12      IG01  trace.to_Tomcat03.duration_mean     0.727930     0.316507             2            2                 1615270860                1615269600              0.529238             0.792494                      26
13      IG01   trace.to_Tomcat03.duration_p95     3.130500     1.569075             2            2                 1615270800                1615269600              2.251098             3.191800                      26
14      IG01      trace.to_Tomcat03.row_count   256.500000    70.250000             2            2                 1615271160                1615270140            197.692308           264.000000                      26
15      IG01  trace.to_Tomcat04.duration_mean     0.732841     0.369155             2            2                 1615270680                1615269600              0.551160             0.829567                      24
16      IG01   trace.to_Tomcat04.duration_p95     3.153010     1.976325             2            2                 1615270980                1615269600              2.337552             3.348800                      24
17      IG01  trace.to_Tomcat02.duration_mean     0.818521     0.393152             2            2                 1615270680                1615270020              0.555663             0.885462                      30
18      IG01   trace.to_Tomcat02.duration_p95     3.948292     1.859505             2            2                 1615270440                1615269840              2.401537             4.447500                      30
19      IG01   trace.to_Tomcat01.duration_p95     3.619640     1.623790             2            2                 1615270440                1615269960              2.355843             4.187000                      30```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

# reuse existing df_trace if present; otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# ensure dt column exists and is timezone-aware UTC+8
tz = pytz.timezone('Asia/Shanghai')
if 'dt' not in df_trace.columns:
    df_trace['dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # coerce to datetime and convert to tz to be safe
    df_trace['dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# define requested groups
requested = [
    ('Tomcat03','trace.self.duration_p95'),
    ('Tomcat03','trace.self.duration_mean'),
    ('Tomcat03','trace.self.row_count'),
    ('IG01','trace.to_Tomcat03.duration_p95'),
    ('IG01','trace.to_Tomcat03.duration_mean'),
    ('IG01','trace.to_Tomcat03.row_count'),
    ('IG01','trace.self.duration_p95'),
    ('IG01','trace.self.duration_mean'),
]

# incident window (inclusive) in UTC+8
start = pd.Timestamp('2021-03-09 14:00:00', tz=tz)
end   = pd.Timestamp('2021-03-09 14:30:00', tz=tz)

# filter to window and to requested combos
df_win = df_trace[(df_trace['dt'] >= start) & (df_trace['dt'] <= end)].copy()
# Keep only rows that match any requested pair
mask = df_win.apply(lambda r: (r['cmdb_id'], r['trace_name']) in requested, axis=1)
df_sel = df_win[mask].copy()

# If empty, prepare empty outputs
if df_sel.empty:
    raw_rows_df = pd.DataFrame(columns=['cmdb_id','trace_name','timestamp','value'])
    summary_df = pd.DataFrame(columns=['cmdb_id','trace_name','total_points_returned','min_value','max_value'])
else:
    # keep only needed columns and ensure ordering by timestamp
    df_sel = df_sel[['cmdb_id','trace_name','timestamp','value']].sort_values(['cmdb_id','trace_name','timestamp']).reset_index(drop=True)

    # For each group, take up to 31 rows (the full window length) in chronological order
    raw_rows_df = df_sel.groupby(['cmdb_id','trace_name'], sort=False).head(31).reset_index(drop=True)

    # summary per group: total_points_returned, min and max value in the returned window
    summary = raw_rows_df.groupby(['cmdb_id','trace_name'], sort=False).agg(
        total_points_returned = ('value','count'),
        min_value = ('value','min'),
        max_value = ('value','max')
    ).reset_index()

    # round numeric min/max for compactness
    summary['min_value'] = summary['min_value'].round(6)
    summary['max_value'] = summary['max_value'].round(6)
    summary_df = summary

# Final compact outputs
raw_rows_df, summary_df

```
Out[7]:
```
summary = (
    "Extracted per-minute trace rows for 8 requested series (window 2021-03-09 14:00–14:30 UTC+8):\n\n"
    "Groups returned (cmdb_id | trace_name | points_returned | min -> max):\n"
    "- IG01 | trace.self.duration_mean: 30 points, 0.356945 -> 0.770811\n"
    "- IG01 | trace.self.duration_p95: 30 points, 1.741000 -> 4.008000\n"
    "- IG01 | trace.to_Tomcat03.duration_mean: 26 points, 0.284852 -> 0.792494\n"
    "- IG01 | trace.to_Tomcat03.duration_p95: 26 points, 1.137600 -> 3.191800\n"
    "- IG01 | trace.to_Tomcat03.row_count: 26 points, 11 -> 264\n"
    "- Tomcat03 | trace.self.duration_mean: 26 points, 0.049853 -> 0.110567\n"
    "- Tomcat03 | trace.self.duration_p95: 26 points, 0.040000 -> 1.030000\n"
    "- Tomcat03 | trace.self.row_count: 26 points, 65 -> 3665\n\n"
    "Total raw rows returned: 216 (chronological per-minute rows grouped by the combinations above).\n"
    "Note: The window covers up to 31 minutes; some series have 30 or 26 points in the data (missing minutes for those series).\n\n"
    "If you want, I can (a) show the full per-minute lists for any particular group, (b) convert timestamps to human-readable UTC+8 datetimes, or (c) plot or further analyze spikes/trends in any of these series."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                trace_name   timestamp        value
0        IG01  trace.self.duration_mean  1615269600     0.356945
1        IG01  trace.self.duration_mean  1615269660     0.411405
2        IG01  trace.self.duration_mean  1615269720     0.466109
3        IG01  trace.self.duration_mean  1615269780     0.543228
4        IG01  trace.self.duration_mean  1615269840     0.440856
..        ...                       ...         ...          ...
211  Tomcat03      trace.self.row_count  1615271100  3665.000000
212  Tomcat03      trace.self.row_count  1615271160  3455.000000
213  Tomcat03      trace.self.row_count  1615271220  3451.000000
214  Tomcat03      trace.self.row_count  1615271280  3102.000000
215  Tomcat03      trace.self.row_count  1615271340  3054.000000

[216 rows x 4 columns],     cmdb_id                       trace_name  total_points_returned  min_value    max_value
0      IG01         trace.self.duration_mean                     30   0.356945     0.770811
1      IG01          trace.self.duration_p95                     30   1.741000     4.008000
2      IG01  trace.to_Tomcat03.duration_mean                     26   0.284852     0.792494
3      IG01   trace.to_Tomcat03.duration_p95                     26   1.137600     3.191800
4      IG01      trace.to_Tomcat03.row_count                     26  11.000000   264.000000
5  Tomcat03         trace.self.duration_mean                     26   0.049853     0.110567
6  Tomcat03          trace.self.duration_p95                     26   0.040000     1.030000
7  Tomcat03             trace.self.row_count                     26  65.000000  3665.000000)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

# reuse existing dataframes if present
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# ensure dt exists and timezone is UTC+8
tz = pytz.timezone('Asia/Shanghai')
for df in (df_trace, df_metric):
    if 'dt' not in df.columns:
        df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# incident window
start = pd.Timestamp('2021-03-09 14:00:00', tz=tz)
end   = pd.Timestamp('2021-03-09 14:30:00', tz=tz)

# requested series list
requests = [
    {'cmdb_id':'Tomcat03','series_type':'trace','series_name':'trace.self.duration_p95'},
    {'cmdb_id':'IG01','series_type':'trace','series_name':'trace.self.duration_p95'},
    {'cmdb_id':'IG01','series_type':'trace','series_name':'trace.to_Tomcat03.duration_p95'},
    {'cmdb_id':'IG01','series_type':'trace','series_name':'trace.to_Tomcat03.duration_mean'},
    {'cmdb_id':'IG01','series_type':'metric','series_name':'JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed'},
]

# precompute global P95 for trace and metric (full series)
trace_p95 = df_trace.groupby(['cmdb_id','trace_name'])['value'].quantile(0.95).reset_index().rename(columns={'value':'global_P95'})
metric_p95 = df_metric.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.95).reset_index().rename(columns={'value':'global_P95'})

# filter windowed data
df_trace_win = df_trace[(df_trace['dt'] >= start) & (df_trace['dt'] <= end)].copy()
df_metric_win = df_metric[(df_metric['dt'] >= start) & (df_metric['dt'] <= end)].copy()

rows = []
for req in requests:
    cmdb = req['cmdb_id']
    stype = req['series_type']
    name = req['series_name']
    if stype == 'trace':
        # get global P95
        gp_row = trace_p95[(trace_p95['cmdb_id']==cmdb) & (trace_p95['trace_name']==name)]
        global_P95 = float(gp_row['global_P95'].iloc[0]) if not gp_row.empty else np.nan
        # window points
        sub = df_trace_win[(df_trace_win['cmdb_id']==cmdb) & (df_trace_win['trace_name']==name)].sort_values('timestamp').reset_index(drop=True)
    else:
        gp_row = metric_p95[(metric_p95['cmdb_id']==cmdb) & (metric_p95['kpi_name']==name)]
        global_P95 = float(gp_row['global_P95'].iloc[0]) if not gp_row.empty else np.nan
        sub = df_metric_win[(df_metric_win['cmdb_id']==cmdb) & (df_metric_win['kpi_name']==name)].sort_values('timestamp').reset_index(drop=True)

    total_points = int(sub.shape[0])
    earliest_ts = pd.NA
    cons_count = 0

    if total_points > 0 and not np.isnan(global_P95):
        mask = sub['value'] >= global_P95
        idxs = np.flatnonzero(mask.values)
        if idxs.size > 0:
            first_idx = int(idxs[0])
            earliest_ts = int(sub.at[first_idx,'timestamp'])
            # count consecutive minutes from first_idx (requires 60s step continuity)
            cons_count = 1
            i = first_idx + 1
            while i < len(sub):
                # check timestamp continuity (60 seconds)
                if int(sub.at[i,'timestamp']) - int(sub.at[i-1,'timestamp']) != 60:
                    break
                if sub.at[i,'value'] >= global_P95:
                    cons_count += 1
                    i += 1
                else:
                    break
    # round global_P95 for compactness (if numeric)
    gp95_disp = round(global_P95, 6) if not np.isnan(global_P95) else pd.NA

    rows.append({
        'cmdb_id': cmdb,
        'series_type': stype,
        'series_name': name,
        'global_P95': gp95_disp,
        'earliest_ge_p95_timestamp': earliest_ts,
        'consecutive_ge_p95_count_from_earliest': int(cons_count),
        'total_points_in_window': total_points
    })

result_df = pd.DataFrame(rows, columns=[
    'cmdb_id','series_type','series_name','global_P95',
    'earliest_ge_p95_timestamp','consecutive_ge_p95_count_from_earliest','total_points_in_window'
])

result_df

```
Out[8]:
```
Summary (results for the 5 requested series; window 2021-03-09 14:00–14:30 UTC+8):

- Tomcat03 — trace.self.duration_p95
  - global P95 (full series): 1.004
  - earliest value >= P95 in window: 2021-03-09 14:05:00 (timestamp 1615269900)
  - consecutive minutes >= P95 starting at that time: 1 minute
  - total points in window: 26

- IG01 — trace.self.duration_p95
  - global P95: 3.293187
  - earliest >= P95 in window: 2021-03-09 14:14:00 (timestamp 1615270440)
  - consecutive minutes >= P95 from that time: 1 minute
  - total points in window: 30

- IG01 — trace.to_Tomcat03.duration_p95
  - global P95: 3.1305
  - earliest >= P95 in window: 2021-03-09 14:20:00 (timestamp 1615270800)
  - consecutive minutes >= P95 from that time: 1 minute
  - total points in window: 26

- IG01 — trace.to_Tomcat03.duration_mean
  - global P95: 0.72793
  - earliest >= P95 in window: 2021-03-09 14:21:00 (timestamp 1615270860)
  - consecutive minutes >= P95 from that time: 1 minute
  - total points in window: 26

- IG01 — JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed (metric)
  - global P95: 57,914,504.0
  - earliest >= P95 in window: 2021-03-09 14:05:00 (timestamp 1615269900)
  - consecutive minutes >= P95 from that time: 26 minutes (i.e., sustained from 14:05 through 14:30)
  - total points in window: 31

Interpretation: trace series show isolated single-minute exceedances of their global P95 within the window, while the IG01 JVM NoHeapMemoryUsed metric shows a sustained run (26 consecutive minutes) above its global P95 starting at 14:05. If you want, I can return the raw per-minute values around each earliest event for deeper inspection.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id series_type                                  series_name    global_P95  earliest_ge_p95_timestamp  consecutive_ge_p95_count_from_earliest  total_points_in_window
0  Tomcat03       trace                      trace.self.duration_p95  1.004000e+00                 1615269900                                       1                      26
1      IG01       trace                      trace.self.duration_p95  3.293187e+00                 1615270440                                       1                      30
2      IG01       trace               trace.to_Tomcat03.duration_p95  3.130500e+00                 1615270800                                       1                      26
3      IG01       trace              trace.to_Tomcat03.duration_mean  7.279300e-01                 1615270860                                       1                      26
4      IG01      metric  JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.791450e+07                 1615269900                                      26                      31```
```